# SQL Server

Azure SQL 提供了一个专用的 [向量数据类型](https://learn.microsoft.com/sql/t-sql/data-types/vector-data-type?view=azuresqldb-current&viewFallbackFrom=sql-server-ver16&tabs=csharp-sample)，可简化在关系数据库中直接创建、存储和查询向量嵌入。这消除了对单独的向量数据库及相关集成的需求，提高了解决方案的安全性，同时降低了整体复杂性。

Azure SQL 是一项强大的服务，集可伸缩性、安全性和高可用性于一体，提供了现代数据库解决方案的所有优势。它利用复杂的查询优化器和企业级功能，在执行传统 SQL 查询的同时执行向量相似性搜索，从而增强数据分析和决策能力。

在此处阅读有关使用 [Azure SQL 数据库构建智能应用程序](https://learn.microsoft.com/azure/azure-sql/database/ai-artificial-intelligence-intelligent-applications?view=azuresql) 的更多信息。

本笔记本将向您展示如何利用此集成的 SQL [向量数据库](https://devblogs.microsoft.com/azure-sql/exciting-announcement-public-preview-of-native-vector-support-in-azure-sql-database/) 来存储文档，并使用余弦（余弦距离）、L2（欧几里得距离）和 IP（内积）执行向量搜索查询，以查找与查询向量接近的文档。

## 设置

安装 `langchain-sqlserver` Python 包。

代码位于一个名为 [langchain-sqlserver](https://github.com/langchain-ai/langchain-azure/tree/main/libs/sqlserver) 的集成包中。

In [ ]:
!pip install langchain-sqlserver==0.1.1

## 凭证

运行此笔记本无需任何凭证，只需确保您已下载 `langchain_sqlserver` 包
如果您想获得一流的自动化模型调用跟踪，您还可以通过取消注释以下内容来设置您的 [LangSmith](https://docs.smith.langchain.com/) API 密钥：

In [ ]:
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")
# os.environ["LANGSMITH_TRACING"] = "true"

## 初始化

In [ ]:
from langchain_sqlserver import SQLServer_VectorStore

在 Azure 门户的数据库设置中查找您的 Azure SQL 数据库连接字符串

更多信息：[连接到 Azure SQL DB - Python](https://learn.microsoft.com/en-us/azure/azure-sql/database/connect-query-python?view=azuresql)

In [11]:
import os

import pyodbc

# Define your SQLServer Connection String
_CONNECTION_STRING = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=<YOUR_DBSERVER>.database.windows.net,1433;"
    "Database=test;"
    "TrustServerCertificate=yes;"
    "Connection Timeout=60;"
    "LongAsMax=yes;"
)

# Connection string can vary:
# "mssql+pyodbc://<username>:<password><servername>/<dbname>?driver=ODBC+Driver+18+for+SQL+Server" -> With Username and Password specified
# "mssql+pyodbc://<servername>/<dbname>?driver=ODBC+Driver+18+for+SQL+Server&Trusted_connection=yes" -> Uses Trusted connection
# "mssql+pyodbc://<servername>/<dbname>?driver=ODBC+Driver+18+for+SQL+Server" -> Uses EntraID connection
# "mssql+pyodbc://<servername>/<dbname>?driver=ODBC+Driver+18+for+SQL+Server&Trusted_connection=no" -> Uses EntraID connection

在此示例中，我们使用 Azure OpenAI 生成嵌入，但您也可以使用 LangChain 中提供的不同嵌入。

您可以通过遵循此[指南](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/create-resource?pivots=web-portal)在 Azure 门户上部署 Azure OpenAI 实例的一个版本。在实例运行后，请确保您拥有实例的名称和密钥。您可以在 Azure 门户中，在实例的“密钥和终结点”部分找到密钥。

In [ ]:
!pip install langchain-openai

In [ ]:
# Import the necessary Libraries
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

# Set your AzureOpenAI details
azure_endpoint = "https://<YOUR_ENDPOINT>.openai.azure.com/"
azure_deployment_name_embedding = "text-embedding-3-small"
azure_deployment_name_chatcompletion = "chatcompletion"
azure_api_version = "2023-05-15"
azure_api_key = "YOUR_KEY"


# Use AzureChatOpenAI for chat completions
llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    azure_deployment=azure_deployment_name_chatcompletion,
    openai_api_version=azure_api_version,
    openai_api_key=azure_api_key,
)

# Use AzureOpenAIEmbeddings for embeddings
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=azure_endpoint,
    azure_deployment=azure_deployment_name_embedding,
    openai_api_version=azure_api_version,
    openai_api_key=azure_api_key,
)

## 管理向量存储

A vector store is a database that stores embeddings and their corresponding metadata.
向量存储是存储嵌入及其对应元数据的数据库。

Vector stores are used to power semantic search and retrieval augmented generation (RAG) applications.
向量存储用于支持语义搜索和检索增强生成 (RAG) 应用。

### Create a vector store
### 创建向量存储

You can create a vector store from a list of documents.
您可以从文档列表中创建向量存储。

```python
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Sample documents
documents = [
    Document(page_content="This is the first document.", metadata={"source": "doc1.txt"}),
    Document(page_content="This document is about a second document.", metadata={"source": "doc2.txt"}),
    Document(page_content="This document is about a third document.", metadata={"source": "doc3.txt"}),
]

# Create a FAISS vector store
vector_store = FAISS.from_documents(documents, OpenAIEmbeddings())
```

### Add documents to a vector store
### 向向量存储添加文档

You can add documents to an existing vector store.
您可以向现有的向量存储添加文档。

```python
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Assume vector_store is an existing FAISS vector store
# 假设 vector_store 是一个现有的 FAISS 向量存储

# New documents to add
new_documents = [
    Document(page_content="This is a new document.", metadata={"source": "doc4.txt"}),
    Document(page_content="Another new document.", metadata={"source": "doc5.txt"}),
]

# Add new documents to the vector store
vector_store.add_documents(new_documents)
```

### Search a vector store
### 搜索向量存储

You can search a vector store for documents that are similar to a given query.
您可以搜索向量存储，查找与给定查询相似的文档。

```python
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

# Sample documents
documents = [
    Document(page_content="This is the first document.", metadata={"source": "doc1.txt"}),
    Document(page_content="This document is about a second document.", metadata={"source": "doc2.txt"}),
    Document(page_content="This document is about a third document.", metadata={"source": "doc3.txt"}),
]

# Create a FAISS vector store
vector_store = FAISS.from_documents(documents, OpenAIEmbeddings())

# Query the vector store
query = "What is this document about?"
results = vector_store.similarity_search(query)

# Print the results
for result in results:
    print(result)
```

### Save and load a vector store
### 保存和加载向量存储

You can save a vector store to disk and load it later.
您可以将向量存储保存到磁盘，并在以后加载它。

```python
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

# Sample documents
documents = [
    Document(page_content="This is the first document.", metadata={"source": "doc1.txt"}),
    Document(page_content="This document is about a second document.", metadata={"source": "doc2.txt"}),
    Document(page_content="This document is about a third document.", metadata={"source": "doc3.txt"}),
]

# Create a FAISS vector store
vector_store = FAISS.from_documents(documents, OpenAIEmbeddings())

# Save the vector store to disk
vector_store.save_local("faiss_index")

# Load the vector store from disk
loaded_vector_store = FAISS.load_local("faiss_index", OpenAIEmbeddings())

# Query the loaded vector store
query = "What is this document about?"
results = loaded_vector_store.similarity_search(query)

# Print the results
for result in results:
    print(result)

In [15]:
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_sqlserver import SQLServer_VectorStore

# Initialize the vector store
vector_store = SQLServer_VectorStore(
    connection_string=_CONNECTION_STRING,
    distance_strategy=DistanceStrategy.COSINE,  # optional, if not provided, defaults to COSINE
    embedding_function=embeddings,  # you can use different embeddings provided in LangChain
    embedding_length=1536,
    table_name="langchain_test_table",  # using table with a custom name
)

### 向向量存储添加条目

In [16]:
## we will use some artificial data for this example
query = [
    "I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.",
    "The candy is just red , No flavor . Just  plan and chewy .  I would never buy them again",
    "Arrived in 6 days and were so stale i could not eat any of the 6 bags!!",
    "Got these on sale for roughly 25 cents per cup, which is half the price of my local grocery stores, plus they rarely stock the spicy flavors. These things are a GREAT snack for my office where time is constantly crunched and sometimes you can't escape for a real meal. This is one of my favorite flavors of Instant Lunch and will be back to buy every time it goes on sale.",
    "If you are looking for a less messy version of licorice for the children, then be sure to try these!  They're soft, easy to chew, and they don't get your hands all sticky and gross in the car, in the summer, at the beach, etc. We love all the flavos and sometimes mix these in with the chocolate to have a very nice snack! Great item, great price too, highly recommend!",
    "We had trouble finding this locally - delivery was fast, no more hunting up and down the flour aisle at our local grocery stores.",
    "Too much of a good thing? We worked this kibble in over time, slowly shifting the percentage of Felidae to national junk-food brand until the bowl was all natural. By this time, the cats couldn't keep it in or down. What a mess. We've moved on.",
    "Hey, the description says 360 grams - that is roughly 13 ounces at under $4.00 per can. No way - that is the approximate price for a 100 gram can.",
    "The taste of these white cheddar flat breads is like a regular cracker - which is not bad, except that I bought them because I wanted a cheese taste.<br /><br />What was a HUGE disappointment? How misleading the packaging of the box is. The photo on the box (I bought these in store) makes it look like it is full of long flatbreads (expanding the length and width of the box). Wrong! The plastic tray that holds the crackers is about 2"
    " smaller all around - leaving you with about 15 or so small flatbreads.<br /><br />What is also bad about this is that the company states they use biodegradable and eco-friendly packaging. FAIL! They used a HUGE box for a ridiculously small amount of crackers. Not ecofriendly at all.<br /><br />Would I buy these again? No - I feel ripped off. The other crackers (like Sesame Tarragon) give you a little<br />more bang for your buck and have more flavor.",
    "I have used this product in smoothies for my son and he loves it. Additionally, I use this oil in the shower as a skin conditioner and it has made my skin look great. Some of the stretch marks on my belly has disappeared quickly. Highly recommend!!!",
    "Been taking Coconut Oil for YEARS.  This is the best on the retail market.  I wish it was in glass, but this is the one.",
]

query_metadata = [
    {"id": 1, "summary": "Good Quality Dog Food"},
    {"id": 8, "summary": "Nasty No flavor"},
    {"id": 4, "summary": "stale product"},
    {"id": 11, "summary": "Great value and convenient ramen"},
    {"id": 5, "summary": "Great for the kids!"},
    {"id": 2, "summary": "yum falafel"},
    {"id": 9, "summary": "Nearly killed the cats"},
    {"id": 6, "summary": "Price cannot be correct"},
    {"id": 3, "summary": "Taste is neutral, quantity is DECEITFUL!"},
    {"id": 7, "summary": "This stuff is great"},
    {"id": 10, "summary": "The reviews don't lie"},
]

In [19]:
vector_store.add_texts(texts=query, metadatas=query_metadata)

[1, 8, 4, 11, 5, 2, 9, 6, 3, 7, 10]

## 查询向量存储
一旦创建了向量存储并添加了相关文档，您很可能希望在链或代理运行时对其进行查询。

执行简单的相似性搜索可以这样做：

In [28]:
# Perform a similarity search between the embedding of the query and the embeddings of the documents
simsearch_result = vector_store.similarity_search("Good reviews", k=3)
print(simsearch_result)

[Document(metadata={'id': 1, 'summary': 'Good Quality Dog Food'}, page_content='I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.'), Document(metadata={'id': 7, 'summary': 'This stuff is great'}, page_content='I have used this product in smoothies for my son and he loves it. Additionally, I use this oil in the shower as a skin conditioner and it has made my skin look great. Some of the stretch marks on my belly has disappeared quickly. Highly recommend!!!'), Document(metadata={'id': 5, 'summary': 'Great for the kids!'}, page_content="If you are looking for a less messy version of licorice for the children, then be sure to try these!  They're soft, easy to chew, and they don't get your hands all sticky and gross in the car, in the summer, at the beach, etc. We love all the fla

### 过滤支持：

向量存储支持一组可以应用于文档元数据字段的过滤器。此功能使开发人员和数据分析师能够优化他们的查询，确保搜索结果准确地符合他们的需求。通过基于特定元数据属性应用过滤器，用户可以限制其搜索范围，只关注最相关的数据子集。

In [29]:
# hybrid search -> filter for cases where id not equal to 1.
hybrid_simsearch_result = vector_store.similarity_search(
    "Good reviews", k=3, filter={"id": {"$ne": 1}}
)
print(hybrid_simsearch_result)

[Document(metadata={'id': 7, 'summary': 'This stuff is great'}, page_content='I have used this product in smoothies for my son and he loves it. Additionally, I use this oil in the shower as a skin conditioner and it has made my skin look great. Some of the stretch marks on my belly has disappeared quickly. Highly recommend!!!'), Document(metadata={'id': 5, 'summary': 'Great for the kids!'}, page_content="If you are looking for a less messy version of licorice for the children, then be sure to try these!  They're soft, easy to chew, and they don't get your hands all sticky and gross in the car, in the summer, at the beach, etc. We love all the flavos and sometimes mix these in with the chocolate to have a very nice snack! Great item, great price too, highly recommend!"), Document(metadata={'id': 3, 'summary': 'Taste is neutral, quantity is DECEITFUL!'}, page_content='The taste of these white cheddar flat breads is like a regular cracker - which is not bad, except that I bought them beca

### 相似度搜索及得分：
如果您想执行相似度搜索并获取相应的得分，可以运行：

In [30]:
simsearch_with_score_result = vector_store.similarity_search_with_score(
    "Not a very good product", k=12
)
print(simsearch_with_score_result)

[(Document(metadata={'id': 3, 'summary': 'Taste is neutral, quantity is DECEITFUL!'}, page_content='The taste of these white cheddar flat breads is like a regular cracker - which is not bad, except that I bought them because I wanted a cheese taste.<br /><br />What was a HUGE disappointment? How misleading the packaging of the box is. The photo on the box (I bought these in store) makes it look like it is full of long flatbreads (expanding the length and width of the box). Wrong! The plastic tray that holds the crackers is about 2 smaller all around - leaving you with about 15 or so small flatbreads.<br /><br />What is also bad about this is that the company states they use biodegradable and eco-friendly packaging. FAIL! They used a HUGE box for a ridiculously small amount of crackers. Not ecofriendly at all.<br /><br />Would I buy these again? No - I feel ripped off. The other crackers (like Sesame Tarragon) give you a little<br />more bang for your buck and have more flavor.'), 0.651

如需查看可在 Azure SQL 向量存储上执行的各种搜索的完整列表，请参阅 [API 参考](https://python.langchain.com/api_reference/sqlserver/index.html)。

### 当您已有想要搜索的嵌入时进行相似性搜索

If you already have embeddings for your data, you can use them to perform similarity searches. This is useful when you want to find items that are similar to a given item or a given query.

如果您已经拥有数据的嵌入，则可以使用它们来执行相似性搜索。当您想查找与给定项目或给定查询相似的项目时，此功能非常有用。

Here's how you can do it:

以下是您可以如何操作：

1.  **Load your embeddings:** You'll need to load your pre-computed embeddings. These are typically stored as a numerical array or matrix.

    1.  **加载您的嵌入：** 您需要加载预先计算好的嵌入。这些通常存储为数值数组或矩阵。

2.  **Choose a similarity metric:** Decide on the metric you want to use to measure similarity. Common choices include:
    *   **Cosine similarity:** Measures the cosine of the angle between two vectors. It's good for high-dimensional data and focuses on the orientation of the vectors rather than their magnitude.
    *   **Euclidean distance:** Measures the straight-line distance between two points in Euclidean space.
    *   **Dot product:** A simpler measure that can be effective when vectors are normalized.

    2.  **选择相似性度量：** 决定您要用于衡量相似性的度量。常见的选择包括：
        *   **余弦相似度：** 衡量两个向量之间夹角的余弦值。它适用于高维数据，并侧重于向量的方向而非其大小。
        *   **欧氏距离：** 衡量欧几里得空间中两点之间的直线距离。
        *   **点积：** 一种更简单的度量，当向量被归一化时可能有效。

3.  **Generate an embedding for your query:** If you're searching for items similar to a specific query (e.g., a text description), you'll need to generate an embedding for that query using the same model that generated your data's embeddings.

    3.  **为您的查询生成嵌入：** 如果您正在搜索与特定查询（例如，文本描述）相似的项目，您需要使用生成数据嵌入的相同模型为该查询生成嵌入。

4.  **Perform the search:** Use a library or framework that supports similarity search with your chosen metric. You'll typically provide your query embedding and your collection of data embeddings. The search will return the most similar items based on the chosen metric.

    4.  **执行搜索：** 使用支持您所选度量的相似性搜索的库或框架。您通常会提供查询嵌入和数据嵌入集合。搜索将根据所选度量返回最相似的项目。

**Example using Python and a hypothetical library:**

**使用 Python 和一个假设的库的示例：**

```python
# Assume you have your embeddings loaded into a NumPy array called 'data_embeddings'
# Assume 'query_text' is the text you want to find similar items for
# Assume 'embedding_model' is the model used to generate embeddings

import numpy as np
# from some_similarity_library import SimilaritySearcher

# 1. Load your embeddings (already done in this assumption)
# data_embeddings = np.load("your_embeddings.npy")

# 3. Generate an embedding for your query
query_embedding = embedding_model.encode(query_text)

# 2. Choose a similarity metric (e.g., cosine similarity)
# 4. Perform the search
# searcher = SimilaritySearcher(data_embeddings, metric="cosine")
# similar_items_indices = searcher.search(query_embedding, k=5) # Find top 5 similar items

# print(f"Indices of the 5 most similar items: {similar_items_indices}")
```

```python
# 假设您已将嵌入加载到名为 'data_embeddings' 的 NumPy 数组中
# 假设 'query_text' 是您要查找相似项的文本
# 假设 'embedding_model' 是用于生成嵌入的模型

import numpy as np
# from some_similarity_library import SimilaritySearcher

# 1. 加载您的嵌入（在此假设中已完成）
# data_embeddings = np.load("your_embeddings.npy")

# 3. 为您的查询生成嵌入
query_embedding = embedding_model.encode(query_text)

# 2. 选择相似性度量（例如，余弦相似度）
# 4. 执行搜索
# searcher = SimilaritySearcher(data_embeddings, metric="cosine")
# similar_items_indices = searcher.search(query_embedding, k=5) # 查找最相似的 5 个项目

# print(f"最相似的 5 个项目的索引：{similar_items_indices}")
```

**Key considerations:**

**关键注意事项：**

*   **Embedding quality:** The quality of your embeddings directly impacts the effectiveness of the similarity search. Ensure your embeddings capture the semantic meaning of your data.
*   **Scalability:** For very large datasets, you might need specialized libraries or vector databases (like FAISS, Annoy, Pinecone, Weaviate) that are optimized for efficient similarity search on massive amounts of data. These often use approximate nearest neighbor (ANN) algorithms.
*   **Metric choice:** The best similarity metric depends on your specific data and use case. Experiment with different metrics to see which one yields the best results.

*   **嵌入质量：** 您的嵌入质量直接影响相似性搜索的有效性。确保您的嵌入能够捕捉数据的语义含义。
*   **可扩展性：** 对于非常大的数据集，您可能需要专门的库或向量数据库（如 FAISS、Annoy、Pinecone、Weaviate），这些库或数据库针对海量数据的高效相似性搜索进行了优化。它们通常使用近似最近邻（ANN）算法。
*   **度量选择：** 最佳相似性度量取决于您的具体数据和用例。尝试不同的度量，看看哪种能产生最佳结果。

In [ ]:
# if you already have embeddings you want to search on
simsearch_by_vector = vector_store.similarity_search_by_vector(
    [-0.0033353185281157494, -0.017689190804958344, -0.01590404286980629, ...]
)
print(simsearch_by_vector)

[Document(metadata={'id': 8, 'summary': 'Nasty No flavor'}, page_content='The candy is just red , No flavor . Just  plan and chewy .  I would never buy them again'), Document(metadata={'id': 4, 'summary': 'stale product'}, page_content='Arrived in 6 days and were so stale i could not eat any of the 6 bags!!'), Document(metadata={'id': 3, 'summary': 'Taste is neutral, quantity is DECEITFUL!'}, page_content='The taste of these white cheddar flat breads is like a regular cracker - which is not bad, except that I bought them because I wanted a cheese taste.<br /><br />What was a HUGE disappointment? How misleading the packaging of the box is. The photo on the box (I bought these in store) makes it look like it is full of long flatbreads (expanding the length and width of the box). Wrong! The plastic tray that holds the crackers is about 2 smaller all around - leaving you with about 15 or so small flatbreads.<br /><br />What is also bad about this is that the company states they use biodegr

In [ ]:
# Similarity Search with Score if you already have embeddings you want to search on
simsearch_by_vector_with_score = vector_store.similarity_search_by_vector_with_score(
    [-0.0033353185281157494, -0.017689190804958344, -0.01590404286980629, ...]
)
print(simsearch_by_vector_with_score)

[(Document(metadata={'id': 8, 'summary': 'Nasty No flavor'}, page_content='The candy is just red , No flavor . Just  plan and chewy .  I would never buy them again'), 0.9648153551769503), (Document(metadata={'id': 4, 'summary': 'stale product'}, page_content='Arrived in 6 days and were so stale i could not eat any of the 6 bags!!'), 0.9655108580341948), (Document(metadata={'id': 3, 'summary': 'Taste is neutral, quantity is DECEITFUL!'}, page_content='The taste of these white cheddar flat breads is like a regular cracker - which is not bad, except that I bought them because I wanted a cheese taste.<br /><br />What was a HUGE disappointment? How misleading the packaging of the box is. The photo on the box (I bought these in store) makes it look like it is full of long flatbreads (expanding the length and width of the box). Wrong! The plastic tray that holds the crackers is about 2 smaller all around - leaving you with about 15 or so small flatbreads.<br /><br />What is also bad about thi

## 从向量存储中删除项目

You can delete items from a vector store by providing the `ids` of the items you want to remove.

您可以通过提供要删除的项目的 `ids` 来从向量存储中删除项目。

```python
from langchain_community.vectorstores import FAISS

# Assuming you have a vector store initialized
# 假设您已经初始化了一个向量存储
vector_store = FAISS.from_texts(
    ["honda is a car company", "apple is a tech company"], {"source": "webpage"}
)

# Delete items by their IDs
# 通过它们的 ID 删除项目
vector_store.delete(["doc1", "doc2"])
```

You can also delete items by providing a `filter` to the `delete` method.

您还可以通过向 `delete` 方法提供 `filter` 来删除项目。

```python
from langchain_community.vectorstores import FAISS

# Assuming you have a vector store initialized
# 假设您已经初始化了一个向量存储
vector_store = FAISS.from_texts(
    ["honda is a car company", "apple is a tech company"], {"source": "webpage"}
)

# Delete items by filter
# 通过过滤器删除项目
vector_store.delete(filter={"source": "webpage"})

### 按 ID 删除行

You can delete a row by its ID.
您可以按 ID 删除行。

```jsx
import { useTable } from '@refinedev/react-table';
import { ColumnDef } from '@tanstack/react-table';
import { useRouter } from 'next/navigation';
import React from 'react';

const columns: ColumnDef<any>[] = [
    {
        id: 'title',
        accessorKey: 'title',
        header: 'Title',
        cell: ({ row }) => row.original.title,
    },
    {
        id: 'content',
        accessorKey: 'content',
        header: 'Content',
        cell: ({ row }) => row.original.content,
    },
    {
        id: 'actions',
        header: 'Actions',
        cell: function Cell({ row }) {
            const { deleteButtonProps } = useTable();

            return (
                <button
                    {...deleteButtonProps({
                        id: row.original.id,
                        // You can also pass other props to the delete button
                        // onClick: () => console.log('Deleting row with id:', row.original.id),
                    })}
                >
                    Delete
                </button>
            );
        },
    },
];

export const BasicTable = () => {
    const {
        getTableProps,
        getTableBodyProps,
        headerGroups,
        rows,
        getState,
        setPageIndex,
        setPageSize,
    } = useTable({
        columns,
        refineCoreProps: {
            // You can also pass other props to the useTable hook
            // queryOptions: {
            //     // staleTime: 10000,
            // },
        },
    });

    return (
        <table>
            <thead {...getTableProps()}>
                {headerGroups.map((headerGroup) => (
                    <tr {...headerGroup.getHeaderGroupProps()}>
                        {headerGroup.headers.map((column) => (
                            <th {...column.getHeaderProps()}>
                                {column.render('Header')}
                            </th>
                        ))}
                    </tr>
                ))}
            </thead>
            <tbody {...getTableBodyProps()}>
                {rows.map((row) => {
                    return (
                        <tr {...row.getRowProps()}>
                            {row.getVisibleCells().map((cell) => {
                                return (
                                    <td {...cell.getCellProps()}>
                                        {cell.render('Cell')}
                                    </td>
                                );
                            })}
                        </tr>
                    );
                })}
            </tbody>
        </table>
    );
};
```

```jsx
import { useTable } from '@refinedev/react-table';
import { ColumnDef } from '@tanstack/react-table';
import { useRouter } from 'next/navigation';
import React from 'react';

const columns: ColumnDef<any>[] = [
    {
        id: 'title',
        accessorKey: 'title',
        header: 'Title',
        cell: ({ row }) => row.original.title,
    },
    {
        id: 'content',
        accessorKey: 'content',
        header: 'Content',
        cell: ({ row }) => row.original.content,
    },
    {
        id: 'actions',
        header: 'Actions',
        cell: function Cell({ row }) {
            const { deleteButtonProps } = useTable();

            return (
                <button
                    {...deleteButtonProps({
                        id: row.original.id,
                        // You can also pass other props to the delete button
                        // onClick: () => console.log('Deleting row with id:', row.original.id),
                    })}
                >
                    Delete
                </button>
            );
        },
    },
];

export const BasicTable = () => {
    const {
        getTableProps,
        getTableBodyProps,
        headerGroups,
        rows,
        getState,
        setPageIndex,
        setPageSize,
    } = useTable({
        columns,
        refineCoreProps: {
            // You can also pass other props to the useTable hook
            // queryOptions: {
            //     // staleTime: 10000,
            // },
        },
    });

    return (
        <table>
            <thead {...getTableProps()}>
                {headerGroups.map((headerGroup) => (
                    <tr {...headerGroup.getHeaderGroupProps()}>
                        {headerGroup.headers.map((column) => (
                            <th {...column.getHeaderProps()}>
                                {column.render('Header')}
                            </th>
                        ))}
                    </tr>
                ))}
            </thead>
            <tbody {...getTableBodyProps()}>
                {rows.map((row) => {
                    return (
                        <tr {...row.getRowProps()}>
                            {row.getVisibleCells().map((cell) => {
                                return (
                                    <td {...cell.getCellProps()}>
                                        {cell.render('Cell')}
                                    </td>
                                );
                            })}
                        </tr>
                    );
                })}
            </tbody>
        </table>
    );
};

In [35]:
# delete row by id
vector_store.delete(["3", "7"])

True

### Dropping Vector Stores

A vector store is a data structure that stores vector embeddings. Vector stores are often used in machine learning and natural language processing applications for tasks such as similarity search, recommendation systems, and anomaly detection.

You can drop a vector store by calling the `drop_vector_store` method on the `VectorStore` object.

```python
from llama_index.vector_stores.types import VectorStore
from llama_index.vector_stores.chroma import ChromaVectorStore

# Assuming you have a ChromaVectorStore instance named 'vector_store'
# For example:
# vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Drop the vector store
vector_store.drop_vector_store()

In [ ]:
# drop vectorstore
vector_store.drop()

## 从 Azure Blob 存储加载文档

You can load documents from Azure Blob Storage into your application. This example shows how to load a document from Azure Blob Storage using the `AzureBlobStorage` class.

您可以将文档从 Azure Blob 存储加载到您的应用程序中。此示例展示了如何使用 `AzureBlobStorage` 类从 Azure Blob 存储加载文档。

```python
from langchain_community.document_loaders import AzureBlobStorageFileLoader

# Replace with your actual Azure Blob Storage details
# 请替换为您实际的 Azure Blob 存储详细信息
connection_string = "YOUR_AZURE_STORAGE_CONNECTION_STRING"
container = "YOUR_CONTAINER_NAME"
blob_name = "YOUR_BLOB_NAME" # e.g., "path/to/your/document.txt"

loader = AzureBlobStorageFileLoader(
    conn_str=connection_string,
    container=container,
    blob_name=blob_name,
)

documents = loader.load()
```

### Parameters

### 参数

*   `conn_str` (str): Your Azure Storage account connection string.
*   `conn_str` (str): 您的 Azure 存储账户连接字符串。
*   `container` (str): The name of the blob container.
*   `container` (str): Blob 容器的名称。
*   `blob_name` (str): The name of the blob (file) to load. This can include a path within the container.
*   `blob_name` (str): 要加载的 blob（文件）的名称。这可以包含容器内的路径。
*   `encoding` (str, optional): The encoding of the file. Defaults to "utf-8".
*   `encoding` (str, optional): 文件的编码。默认为 "utf-8"。
*   `metadata` (Callable, optional): A function that takes the blob name and returns metadata for the document. Defaults to None.
*   `metadata` (Callable, optional): 一个函数，它接受 blob 名称并返回文档的元数据。默认为 None。
*   `client_kwargs` (dict, optional): Additional keyword arguments to pass to the Azure Blob Storage client. Defaults to None.
*   `client_kwargs` (dict, optional): 要传递给 Azure Blob 存储客户端的其他关键字参数。默认为 None。

以下是将文件从 Azure Blob Storage 容器加载到 SQL Vector 存储的示例，该文件在被分割成块之后。
[Azure Blob Storage](https://learn.microsoft.com/en-us/azure/storage/blobs/storage-blobs-introduction) 是 Microsoft 的云对象存储解决方案。Blob Storage 针对存储海量非结构化数据进行了优化。

In [ ]:
pip install azure-storage-blob

In [36]:
from langchain.document_loaders import AzureBlobStorageFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Define your connection string and blob details
conn_str = "DefaultEndpointsProtocol=https;AccountName=<YourBlobName>;AccountKey=<YourAccountKey>==;EndpointSuffix=core.windows.net"
container_name = "<YourContainerName"
blob_name = "01 Harry Potter and the Sorcerers Stone.txt"

# Create an instance of AzureBlobStorageFileLoader
loader = AzureBlobStorageFileLoader(
    conn_str=conn_str, container=container_name, blob_name=blob_name
)

# Load the document from Azure Blob Storage
documents = loader.load()

# Split the document into smaller chunks if necessary
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_documents = text_splitter.split_documents(documents)

# Print the number of split documents
print(f"Number of split documents: {len(split_documents)}")

Number of split documents: 528


API 参考：[AzureBlobStorageContainerLoader](https://python.langchain.com/api_reference/community/document_loaders/langchain_community.document_loaders.azure_blob_storage_container.AzureBlobStorageContainerLoader.html)

In [39]:
# # Initialize the vector store & insert the documents in AzureSQLDB with their embeddings
vector_store = SQLServer_VectorStore(
    connection_string=_CONNECTION_STRING,
    distance_strategy=DistanceStrategy.COSINE,
    embedding_function=embeddings,
    embedding_length=1536,
    table_name="harrypotter",
)  # Replace with your actual vector store initialization

# Add split documents to the vector store individually
for i, doc in enumerate(split_documents):
    vector_store.add_documents(documents=[doc], ids=[f"doc_{i}"])

print("Documents added to the vector store successfully!")

Documents added to the vector store successfully!


## 直接查询

In [46]:
from typing import List, Tuple

# Perform similarity search
query = "Why did the Dursleys not want Harry in their house?"
docs_with_score: List[Tuple[Document, float]] = (
    vector_store.similarity_search_with_score(query)
)

for doc, score in docs_with_score:
    print("-" * 60)
    print("Score: ", score)
    print(doc.page_content)
    print("-" * 60)

------------------------------------------------------------
Score:  0.3626232679001803
The Dursleys had everything they wanted, but they also had a secret, and their greatest fear was that somebody would discover it. They didn’t think they could bear it if anyone found out about the Potters. Mrs. Potter was Mrs. Dursley’s sister, but they hadn’t met for several years; in fact, Mrs. Dursley pretended she didn’t have a sister, because her sister and her good-for-nothing husband were as unDursleyish as it was possible to be. The Dursleys shuddered to think what the neighbors would say if the Potters arrived in the street. The Dursleys knew that the Potters had a small son, too, but they had never even seen him. This boy was another good reason for keeping the Potters away; they didn’t want Dudley mixing with a child like that.
------------------------------------------------------------
------------------------------------------------------------
Score:  0.44752797298657554
The Dursleys’

## 用于检索增强生成的使用方法

#### 用例 1：基于故事书的问答系统

问答功能允许用户就故事、角色和事件提出具体问题，并获得简洁、富含上下文的答案。这不仅能加深他们对书籍的理解，还能让他们感觉自己是魔法宇宙的一部分。

## 通过转换为检索器进行查询

LangChain 向量存储通过实现高效的相似性搜索，根据用户查询找到最相关的 10 个文档，从而简化了复杂问答系统的构建。**检索器**是从 **vector\_store** 创建的，而问答链则使用 **create\_stuff\_documents\_chain** 函数构建。使用 **ChatPromptTemplate** 类精心制作提示模板，确保响应结构化且富含上下文。在问答应用中，向用户展示用于生成答案的来源通常很重要。LangChain 内置的 **create\_retrieval\_chain** 会将检索到的源文档作为“context”键传播到输出中：

在此处阅读有关 Langchain RAG 教程及上述术语的更多信息：[https:/python.langchain.com/docs/tutorials/rag](https:/python.langchain.com/docs/tutorials/rag)

In [ ]:
from typing import List, Tuple

import pandas as pd
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


# Define the function to perform the RAG chain invocation
def get_answer_and_sources(user_query: str):
    # Perform similarity search with scores
    docs_with_score: List[Tuple[Document, float]] = (
        vector_store.similarity_search_with_score(
            user_query,
            k=10,
        )
    )

    # Extract the context from the top results
    context = "\n".join([doc.page_content for doc, score in docs_with_score])

    # Define the system prompt
    system_prompt = (
        "You are an assistant for question-answering tasks based on the story in the book. "
        "Use the following pieces of retrieved context to answer the question. "
        "If you don't know the answer, say that you don't know, but also suggest that the user can use the fan fiction function to generate fun stories. "
        "Use 5 sentences maximum and keep the answer concise by also providing some background context of 1-2 sentences."
        "\n\n"
        "{context}"
    )

    # Create the prompt template
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}"),
        ]
    )

    # Create the retriever and chains
    retriever = vector_store.as_retriever()
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    # Define the input
    input_data = {"input": user_query}

    # Invoke the RAG chain
    response = rag_chain.invoke(input_data)

    # Print the answer
    print("Answer:", response["answer"])

    # Prepare the data for the table
    data = {
        "Doc ID": [
            doc.metadata.get("source", "N/A").split("/")[-1]
            for doc in response["context"]
        ],
        "Content": [
            doc.page_content[:50] + "..."
            if len(doc.page_content) > 100
            else doc.page_content
            for doc in response["context"]
        ],
    }

    # Create a DataFrame
    df = pd.DataFrame(data)

    # Print the table
    print("\nSources:")
    print(df.to_markdown(index=False))

In [1]:
# Define the user query
user_query = "How did Harry feel when he first learnt that he was a Wizard?"

# Call the function to get the answer and sources
get_answer_and_sources(user_query)

Answer: When Harry first learned that he was a wizard, he felt quite sure there had been a horrible mistake. He struggled to believe it because he had spent his life being bullied and mistreated by the Dursleys. If he was really a wizard, he wondered why he hadn't been able to use magic to defend himself. This disbelief and surprise were evident when he gasped, “I’m a what?”

Sources:
| Doc ID                                      | Content                                               |
|:--------------------------------------------|:------------------------------------------------------|
| 01 Harry Potter and the Sorcerers Stone.txt | Harry was wondering what a wizard did once he’d fi... |
| 01 Harry Potter and the Sorcerers Stone.txt | Harry realized his mouth was open and closed it qu... |
| 01 Harry Potter and the Sorcerers Stone.txt | “Most of us reckon he’s still out there somewhere ... |
| 01 Harry Potter and the Sorcerers Stone.txt | “Ah, go boil yer heads, both of yeh,” said H

In [2]:
# Define the user query
user_query = "Did Harry have a pet? What was it"

# Call the function to get the answer and sources
get_answer_and_sources(user_query)

Yes, Harry had a pet owl named Hedwig. He decided to call her Hedwig after finding the name in a book titled *A History of Magic*.

Sources:
| Doc ID                                      | Content                                               |
|:--------------------------------------------|:------------------------------------------------------|
| 01 Harry Potter and the Sorcerers Stone.txt | Harry sank down next to the bowl of peas. “What di... |
| 01 Harry Potter and the Sorcerers Stone.txt | Harry kept to his room, with his new owl for compa... |
| 01 Harry Potter and the Sorcerers Stone.txt | As the snake slid swiftly past him, Harry could ha... |
| 01 Harry Potter and the Sorcerers Stone.txt | Ron reached inside his jacket and pulled out a fat... |



## API 参考

有关 SQLServer Vectorstore 功能和配置的详细文档，请访问 API 参考：[https://python.langchain.com/api\_reference/sqlserver/index.html](https://python.langchain.com/api\_reference/sqlserver/index.html)

## 相关
- 向量存储 [概念指南](https://python.langchain.com/docs/concepts/vectorstores/)
- 向量存储 [操作指南](https://python.langchain.com/docs/how_to/#vector-stores)